# Agriculture

In [6]:
!pip cache purge --quiet

In [7]:
!pip install geopandas==1.1.1 --quiet

In [8]:
import base64
import geopandas as gpd
import mimetypes
import numpy as np
import pandas as pd
import random

from IPython.display import HTML
from shapely.geometry import Point

In [9]:
# If you've forked this repo, change OWNER to your GitHub username.
# REPO and BRANCH will normally stay the same unless you renamed them.
OWNER = "singlestore-cookbook"
REPO = "singlestore-cookbook.github.io"
BRANCH = "refs/heads/main"

BASE_URL = f"https://raw.githubusercontent.com/{OWNER}/{REPO}/{BRANCH}/code/part-ml/agriculture-analytics-using-r/datasets"

In [10]:
# Load GeoJSON of provinces
geojson_url = f"{BASE_URL}/geoBoundaries-IDN-ADM1.geojson"

gdf = gpd.read_file(geojson_url)

In [11]:
# Map province name to polygon
province_polygons = {row['shapeName']: row['geometry'] for _, row in gdf.iterrows()}

def random_point_within(poly):
    minx, miny, maxx, maxy = poly.bounds
    while True:
        p = Point(random.uniform(minx, maxx), random.uniform(miny, maxy))
        if poly.contains(p):
            return p

In [12]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [13]:
# Major crops per province (alphabetical by province)
province_crop_weights = {
    "Aceh": {"Rice": 0.6, "Corn": 0.2, "Coffee": 0.2},
    "Bali": {"Rice": 0.5, "Coffee": 0.3, "Fruit": 0.2},
    "Bangka-Belitung Islands": {"Rice": 0.3, "Pepper": 0.4, "Rubber": 0.3},
    "Banten": {"Rice": 0.5, "Corn": 0.3, "Coconut": 0.2},
    "Bengkulu": {"Rice": 0.4, "Coffee": 0.4, "Cinnamon": 0.2},
    "Central Java": {"Rice": 0.6, "Corn": 0.3, "Sugarcane": 0.1},
    "Central Kalimantan": {"Rice": 0.3, "Palm Oil": 0.4, "Rubber": 0.3},
    "Central Sulawesi": {"Rice": 0.3, "Cocoa": 0.5, "Corn": 0.2},
    "East Java": {"Rice": 0.5, "Corn": 0.3, "Tobacco": 0.2},
    "East Kalimantan": {"Rice": 0.3, "Palm Oil": 0.4, "Rubber": 0.3},
    "East Nusa Tenggara": {"Cassava": 0.3, "Corn": 0.3, "Rice": 0.2, "Cocoa": 0.2},
    "Gorontalo": {"Cashew": 0.3, "Corn": 0.3, "Rice": 0.3, "Soybeans": 0.1},
    "Jakarta Special Capital Region": {"Fruit": 0.6, "Urban Vegetables": 0.4},
    "Jambi": {"Coffee": 0.3, "Rice": 0.3, "Rubber": 0.4},
    "Lampung": {"Coffee": 0.3, "Rice": 0.3, "Sugarcane": 0.4},
    "Maluku": {"Cloves": 0.4, "Coconut": 0.4, "Rice": 0.2},
    "North Kalimantan": {"Palm Oil": 0.4, "Rice": 0.3, "Rubber": 0.3},
    "North Maluku": {"Coconut": 0.3, "Nutmeg": 0.5, "Rice": 0.2},
    "North Sulawesi": {"Cashew": 0.2, "Cloves": 0.3, "Coconut": 0.3, "Rice": 0.2},
    "North Sumatra": {"Palm Oil": 0.4, "Rice": 0.3, "Rubber": 0.3},
    "Papua": {"Cassava": 0.3, "Rice": 0.2, "Sago": 0.3, "Sweet Potato": 0.2},
    "Riau": {"Coconut": 0.3, "Palm Oil": 0.4, "Rubber": 0.3},
    "Riau Islands": {"Coconut": 0.7, "Rice": 0.3},
    "South Kalimantan": {"Palm Oil": 0.4, "Rice": 0.3, "Rubber": 0.3},
    "South Sulawesi": {"Cashew": 0.2, "Cocoa": 0.4, "Corn": 0.2, "Rice": 0.2},
    "South Sumatra": {"Palm Oil": 0.4, "Rice": 0.3, "Rubber": 0.3},
    "Southeast Sulawesi": {"Cashew": 0.3, "Cocoa": 0.4, "Rice": 0.3},
    "Special Region of Yogyakarta": {"Corn": 0.3, "Rice": 0.4, "Soybeans": 0.3},
    "West Java": {"Coffee": 0.2, "Rice": 0.5, "Tea": 0.3},
    "West Kalimantan": {"Palm Oil": 0.4, "Rice": 0.3, "Rubber": 0.3},
    "West Nusa Tenggara": {"Corn": 0.4, "Rice": 0.4, "Soybeans": 0.2},
    "West Papua": {"Cassava": 0.3, "Rice": 0.2, "Sago": 0.3, "Sweet Potato": 0.2},
    "West Sulawesi": {"Cocoa": 0.4, "Coconut": 0.3, "Rice": 0.3},
    "West Sumatra": {"Cinnamon": 0.3, "Rice": 0.4, "Tea": 0.3}
}

# Rainfall per province (alphabetical by province, mm/year)
province_rainfall = {
    "Aceh": 2500,
    "Bali": 2100,
    "Bangka-Belitung Islands": 2100,
    "Banten": 2000,
    "Bengkulu": 2300,
    "Central Java": 2800,
    "Central Kalimantan": 3000,
    "Central Sulawesi": 2400,
    "East Java": 2500,
    "East Kalimantan": 2800,
    "East Nusa Tenggara": 1350,
    "Gorontalo": 2300,
    "Jakarta Special Capital Region": 1800,
    "Jambi": 2400,
    "Lampung": 2000,
    "Maluku": 3000,
    "North Kalimantan": 2700,
    "North Maluku": 2800,
    "North Sulawesi": 2500,
    "North Sumatra": 2800,
    "Papua": 3600,
    "Riau": 2600,
    "Riau Islands": 2500,
    "South Kalimantan": 2900,
    "South Sulawesi": 2200,
    "South Sumatra": 2200,
    "Southeast Sulawesi": 2100,
    "Special Region of Yogyakarta": 2700,
    "West Java": 3000,
    "West Kalimantan": 3200,
    "West Nusa Tenggara": 1800,
    "West Papua": 3500,
    "West Sulawesi": 2200,
    "West Sumatra": 2700
}

# Crop soil pH ranges
crop_ph_ranges = {
    "Cashew": (5.5, 6.5),
    "Cassava": (4.5, 6.5),
    "Cinnamon": (4.5, 5.5),
    "Cloves": (5.0, 6.0),
    "Cocoa": (5.0, 6.0),
    "Coconut": (5.5, 7.5),
    "Coffee": (5.0, 6.5),
    "Corn": (5.5, 7.0),
    "Fruit": (5.5, 6.5),
    "Nutmeg": (5.5, 6.5),
    "Palm Oil": (4.5, 6.0),
    "Pepper": (5.5, 6.5),
    "Rice": (5.0, 6.5),
    "Rubber": (4.5, 6.0),
    "Sago": (4.5, 6.0),
    "Soybeans": (6.0, 7.0),
    "Sugarcane": (6.0, 7.0),
    "Sweet Potato": (5.5, 6.5),
    "Tea": (4.5, 5.5),
    "Tobacco": (5.5, 6.5),
    "Urban Vegetables": (6.0, 7.0)
}

# Base yield per crop (tons/ha)
base_yield = {
    "Cashew": 1.0,
    "Cassava": 10.0,
    "Cinnamon": 0.8,
    "Cloves": 1.0,
    "Cocoa": 1.8,
    "Coconut": 4.0,
    "Coffee": 1.5,
    "Corn": 4.5,
    "Fruit": 4.0,
    "Nutmeg": 1.0,
    "Palm Oil": 4.0,
    "Pepper": 1.2,
    "Rice": 5.0,
    "Rubber": 1.0,
    "Sago": 12.0,
    "Soybeans": 2.5,
    "Sugarcane": 7.0,
    "Sweet Potato": 15.0,
    "Tea": 2.0,
    "Tobacco": 2.0,
    "Urban Vegetables": 5.0
}

# Regional trend categories
province_trend_categories = {
    "Aceh": "vulnerable",
    "Bali": "resilient",
    "Bangka-Belitung Islands": "vulnerable",
    "Banten": "coastal",
    "Bengkulu": "vulnerable",
    "Central Java": "resilient",
    "Central Kalimantan": "stable",
    "Central Sulawesi": "vulnerable",
    "East Java": "resilient",
    "East Kalimantan": "stable",
    "East Nusa Tenggara": "vulnerable",
    "Gorontalo": "vulnerable",
    "Jakarta Special Capital Region": "vulnerable",
    "Jambi": "vulnerable",
    "Lampung": "coastal",
    "Maluku": "vulnerable",
    "North Kalimantan": "stable",
    "North Maluku": "vulnerable",
    "North Sulawesi": "vulnerable",
    "North Sumatra": "coastal",
    "Papua": "stable",
    "Riau": "coastal",
    "Riau Islands": "coastal",
    "South Kalimantan": "stable",
    "South Sulawesi": "coastal",
    "South Sumatra": "coastal",
    "Southeast Sulawesi": "vulnerable",
    "Special Region of Yogyakarta": "resilient",
    "West Java": "resilient",
    "West Kalimantan": "stable",
    "West Nusa Tenggara": "vulnerable",
    "West Papua": "stable",
    "West Sulawesi": "vulnerable",
    "West Sumatra": "vulnerable"
}

# Trend adjustments and rates per category
trend_params = {
    "coastal": {"rain_adj": -20, "tech_decline": 0.007, "ph_drift": -0.05},
    "stable": {"rain_adj": -5, "tech_decline": 0.007, "ph_drift": -0.02},
    "resilient": {"rain_adj": -3, "tech_decline": 0.003, "ph_drift": -0.01},
    "vulnerable": {"rain_adj": -10, "tech_decline": 0.007, "ph_drift": -0.05},
}

In [14]:
def weighted_choice(weight_dict):
    crops, weights = zip(*weight_dict.items())
    return random.choices(crops, weights = weights, k = 1)[0]

def generate_row():
    # Error handling for missing polygons
    try:
        province = random.choice(list(province_polygons.keys()))
        crop = weighted_choice(province_crop_weights[province])
        polygon = province_polygons[province]
    except KeyError:
        print("Error: Province data missing from province_polygons dictionary.")
        return None
    except IndexError:
        print("Error: province_polygons dictionary is empty.")
        return None

    # Generate random lat/lon inside polygon
    for _ in range(10):  # retry loop
        minx, miny, maxx, maxy = polygon.bounds
        lat = random.uniform(miny, maxy)
        lon = random.uniform(minx, maxx)
        point = Point(lon, lat)
        if polygon.contains(point):
            break
    else:
        return None

    # Year
    year = random.randint(2015, 2024)

    # Get trend parameters
    trend_category = province_trend_categories.get(province, "vulnerable")
    params = trend_params.get(trend_category, trend_params["vulnerable"])

    # Rainfall with province-specific trend
    base_rain = province_rainfall[province]
    rainfall = int(np.clip(
        np.random.normal(base_rain + max(0, year - 2020) * params["rain_adj"], 200),
        base_rain * 0.7,
        base_rain * 1.3
    ))

    # Soil pH with trend
    ph_low, ph_high = crop_ph_ranges.get(crop, (5.0, 6.5))
    base_ph = (ph_low + ph_high) / 2
    ph_drift = (year - 2015) * params["ph_drift"]
    soil_ph = round(np.random.normal(base_ph + ph_drift, 0.3), 2)
    soil_ph = np.clip(soil_ph, ph_low, ph_high)

    # Yield calculation
    base = base_yield.get(crop, 3.0)

    # Soil effect
    ph_effect = max(0.8, 1 - abs(soil_ph - base_ph) * 0.1)

    # Rainfall effect
    rainfall_effect = max(0.8, 1 - abs(rainfall - base_rain) / 2000)

    # Tech trend + resilience
    if year <= 2020:
        tech_factor = 1 + (year - 2015) * 0.01
    else:
        tech_factor = 1.05 - (year - 2020) * params["tech_decline"]
        tech_factor = max(0.85, tech_factor)

    # Final yield
    yield_val = base * ph_effect * rainfall_effect * tech_factor * random.uniform(0.9, 1.1)
    yield_val = round(yield_val, 2)

    return {
        "province": province,
        "latitude": round(point.y, 4),
        "longitude": round(point.x, 4),
        "rainfall": rainfall,
        "soil_ph": soil_ph,
        "crop": crop,
        "yield": yield_val,
        "year": year
    }

In [15]:
# Generate dataset
rows = [generate_row() for _ in range(10000)]
df = pd.DataFrame([r for r in rows if r is not None])

In [16]:
df.head()

,province,latitude,longitude,rainfall,soil_ph,crop,yield,year
0,West Kalimantan,0.7313,112.3939,3284,5.05,Palm Oil,3.55,2023
1,Riau Islands,-0.1721,104.6600,2549,6.51,Coconut,3.61,2024
2,Southeast Sulawesi,-3.7564,122.4373,2033,5.33,Rice,4.88,2022
3,Bali,-8.2479,114.5624,2412,5.92,Coffee,1.20,2021
4,East Nusa Tenggara,-8.4012,123.4993,1226,5.51,Rice,4.36,2023


In [17]:
# Save CSV
agriculture_csv = "agriculture.csv"
df.to_csv(agriculture_csv, index = False)

In [18]:
mime_type, _ = mimetypes.guess_type(agriculture_csv)

with open(agriculture_csv, "rb") as f:
    data = f.read()
b64 = base64.b64encode(data).decode()
href = f'<a download="{agriculture_csv}" href="data:{mime_type};base64,{b64}">Download {agriculture_csv}</a>'
HTML(href)